<a href="https://colab.research.google.com/github/AdityaNair-IND/Xfinity_Business_Analysis/blob/main/SynthDatasetXfinity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install faker
import pandas as pd
import numpy as np
import uuid
from faker import Faker
from datetime import datetime, timedelta
from scipy.stats import truncnorm, beta, lognorm, gamma

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 24.1 MB/s eta 0:00:00


In [ ]:
fake = Faker('en_US')
Faker.seed(2025)
np.random.seed(2025)

In [ ]:
class Config:
    NUM_SUBSCRIBERS = 30000
    START_DATE = datetime(2024, 1, 1)
    END_DATE = datetime(2025, 12, 31)
    MONTH_LIST = pd.date_range(start=START_DATE, end=END_DATE, freq='MS').strftime('%Y-%m').tolist()


    PRICING = {
        'UNL_INTRO': {'base': 40.0, 'multi_line': {2: 30.0, 3: 25.0, 4: 25.0}, 'qci': 9},
        'UNL_PREM': {'base': 50.0, 'multi_line': {2: 30.0, 3: 30.0, 4: 30.0}, 'qci': 8},
        'BTG_2025': {'base': 20.0, 'per_gb': 20.0, 'pool_limit': 1.0, 'qci': 8}
    }


    CHURN_BASE_RATE = 0.012
    WEIGHT_LEMON = 0.60
    WEIGHT_BILL_SHOCK = 0.15
    WEIGHT_LOCK_IN = -0.008

In [ ]:
def calculate_rewards_tier(internet_tenure_mos, service_count):
    tenure_yrs = internet_tenure_mos / 12
    if tenure_yrs >= 10: t_score = 3
    elif tenure_yrs >= 5: t_score = 2
    elif tenure_yrs >= 1: t_score = 1
    else: t_score = 0
    if service_count >= 4: s_score = 3
    elif service_count == 3: s_score = 2
    elif service_count == 2: s_score = 1
    else: s_score = 0
    final_score = max(t_score, s_score)
    return ['Silver', 'Gold', 'Platinum', 'Diamond'][final_score]

In [ ]:
def generate_subscribers():
    n = Config.NUM_SUBSCRIBERS
    internet_tenure = np.clip(np.random.gamma(2.0, 48.0, n), 1, 360).astype(int)
    mobile_ratios = np.random.beta(2, 5, n)
    mobile_tenure = (internet_tenure * mobile_ratios).astype(int)
    bundle_opts = ['Double_Play', 'Triple_Play', 'Quad_Play']
    bundles = np.random.choice(bundle_opts, size=n, p=[0.6, 0.3, 0.1])
    svc_map = {'Double_Play': 2, 'Triple_Play': 3, 'Quad_Play': 4}
    svc_counts = [svc_map[b] for b in bundles]
    rewards = [calculate_rewards_tier(it, sc) for it, sc in zip(internet_tenure, svc_counts)]
    lines = np.random.choice([1, 2, 3, 4, 5], size=n, p=[0.4, 0.35, 0.15, 0.08, 0.02])
    credits = np.random.choice(['Prime', 'Subprime', 'No_Hit'], size=n, p=[0.7, 0.2, 0.1])


    return pd.DataFrame({
        'Sub_ID': [str(uuid.uuid4()) for _ in range(n)],
        'Internet_Tenure_Mos': internet_tenure,
        'Mobile_Tenure_Mos': mobile_tenure,
        'Service_Bundle': bundles,
        'Service_Count': svc_counts,
        'Rewards_Tier': rewards,
        'Line_Count': lines,
        'Zip_Code': [fake.zipcode() for _ in range(n)],
        'Credit_Class': credits
    })


In [ ]:
def assign_plans(df):
    plans = []
    for _, row in df.iterrows():
        if row['Line_Count'] >= 3 and row['Credit_Class'] == 'Prime':
            plans.append('UNL_PREM')
        elif row['Line_Count'] == 1 and np.random.rand() < 0.4:
            plans.append('BTG_2025')
        else:
            plans.append('UNL_INTRO')
    df['Plan_ID'] = plans
    return df

In [ ]:
def assign_plans(df):
    plans = []
    for _, row in df.iterrows():
        if row['Line_Count'] >= 3 and row['Credit_Class'] == 'Prime':
            plans.append('UNL_PREM')
        elif row['Line_Count'] == 1 and np.random.rand() < 0.4:
            plans.append('BTG_2025')
        else:
            plans.append('UNL_INTRO')
    df['Plan_ID'] = plans
    return df

In [ ]:
def generate_usage(df):
    records = []
    for _, row in df.iterrows():
        for month in Config.MONTH_LIST:
            plan = row['Plan_ID']
            total_data = lognorm.rvs(s=0.8, scale=22)
            wifi_pct = min(beta.rvs(8, 1.5), 0.99)
            cell_data = total_data * (1 - wifi_pct)
            bill_amt, overage, shock = 0, 0, 0
            lines = row['Line_Count']


            if plan == 'BTG_2025':
                bill_amt = 20
                if cell_data > 1.0:
                    excess = np.ceil(cell_data - 1.0)
                    overage = excess * 20
                    bill_amt += overage
                    if 1.0 < cell_data < 1.5:
                        shock = 1
            elif plan == 'UNL_INTRO':
                bill_amt = 40 + 20 * (lines - 1)
            elif plan == 'UNL_PREM':
                bill_amt = 50 + 30 * (lines - 1)


            records.append({
                'Sub_ID': row['Sub_ID'],
                'Month_ID': month,
                'Total_Data_GB': round(total_data, 2),
                'WiFi_Offload_Pct': round(wifi_pct, 3),
                'Cellular_Data_GB': round(cell_data, 2),
                'Bill_Amount': round(bill_amt, 2),
                'Overage_Charge': round(overage, 2),
                'Bill_Shock': shock
            })
    return pd.DataFrame(records)

In [ ]:
def generate_events(df):
    events = []
    for _, row in df.iterrows():
        if row['Mobile_Tenure_Mos'] <= 3 and np.random.rand() < 0.05:
            events.append({
                'Sub_ID': row['Sub_ID'],
                'Event_Type': 'Port_In_Failure',
                'Event_Outcome': np.random.choice(['Resolved', 'Abandoned'], p=[0.7, 0.3]),
                'Resolution_Hrs': np.random.uniform(24, 96)
            })
    return pd.DataFrame(events)

In [ ]:
def calculate_churn(df_subs, df_usage, df_events):
    df_latest = df_usage[df_usage['Month_ID'] == Config.MONTH_LIST[-1]]
    master = df_subs.merge(df_latest, on='Sub_ID')
    lemons = set(df_events[df_events['Event_Outcome'] == 'Abandoned']['Sub_ID'])
    labels, reasons = [], []


    for _, row in master.iterrows():
        prob = Config.CHURN_BASE_RATE
        reason = 'None'
        if row['Sub_ID'] in lemons:
            prob += Config.WEIGHT_LEMON
            reason = 'Activation_Failure'
        elif row['Bill_Shock'] == 1:
            prob += Config.WEIGHT_BILL_SHOCK
            reason = 'Bill_Shock'
        elif row['Rewards_Tier'] in ['Platinum', 'Diamond']:
            prob *= 0.6
        if row['Mobile_Tenure_Mos'] < 36:
            prob += Config.WEIGHT_LOCK_IN
        churned = int(np.random.rand() < prob)
        labels.append(churned)
        reasons.append(reason if churned else None)


    return pd.DataFrame({
        'Sub_ID': master['Sub_ID'],
        'Churn_Flag': labels,
        'Churn_Reason': reasons
    })

In [ ]:
def run_pipeline():
    df_subs = generate_subscribers()
    df_subs = assign_plans(df_subs)
    df_usage = generate_usage(df_subs)
    df_events = generate_events(df_subs)
    df_churn = calculate_churn(df_subs, df_usage, df_events)


    # Create Master Export
    df_master = df_usage.merge(df_subs, on='Sub_ID', how='left')\
    .merge(df_churn, on='Sub_ID', how='left')\
    .merge(df_events.groupby('Sub_ID').agg({
    'Event_Type': 'first',
    'Event_Outcome': 'first',
    'Resolution_Hrs': 'mean'
    }).reset_index(), on='Sub_ID', how='left')
    df_master['Churn_Flag'] = df_master['Churn_Flag'].fillna(0).astype(int)
    df_master['Churn_Reason'] = df_master['Churn_Reason'].fillna('None')


    df_master.to_excel('xfinity_mobile_synthetic_2024_2025.xlsx', index=False)
    print("\n📦 Excel file 'xfinity_mobile_synthetic_2024_2025.xlsx' saved.")
    return df_master

In [ ]:
master_df = run_pipeline()
master_df.head()


📦 Excel file 'xfinity_mobile_synthetic_2024_2025.xlsx' saved.


,Sub_ID,Month_ID,Total_Data_GB,WiFi_Offload_Pct,Cellular_Data_GB,Bill_Amount,Overage_Charge,Bill_Shock,Internet_Tenure_Mos,Mobile_Tenure_Mos,...,Rewards_Tier,Line_Count,Zip_Code,Credit_Class,Plan_ID,Churn_Flag,Churn_Reason,Event_Type,Event_Outcome,Resolution_Hrs
0,2741dc40-f5e5-4b3b-8383-41d64c1e12c5,2024-01,21.17,0.905,2.02,60.0,0.0,0,190,19,...,Diamond,2,67491,Prime,UNL_INTRO,0,None,NaN,NaN,NaN
1,2741dc40-f5e5-4b3b-8383-41d64c1e12c5,2024-02,14.14,0.966,0.48,60.0,0.0,0,190,19,...,Diamond,2,67491,Prime,UNL_INTRO,0,None,NaN,NaN,NaN
2,2741dc40-f5e5-4b3b-8383-41d64c1e12c5,2024-03,160.10,0.805,31.28,60.0,0.0,0,190,19,...,Diamond,2,67491,Prime,UNL_INTRO,0,None,NaN,NaN,NaN
3,2741dc40-f5e5-4b3b-8383-41d64c1e12c5,2024-04,16.14,0.671,5.31,60.0,0.0,0,190,19,...,Diamond,2,67491,Prime,UNL_INTRO,0,None,NaN,NaN,NaN
4,2741dc40-f5e5-4b3b-8383-41d64c1e12c5,2024-05,17.64,0.770,4.06,60.0,0.0,0,190,19,...,Diamond,2,67491,Prime,UNL_INTRO,0,None,NaN,NaN,NaN


In [ ]:
for col in master_df.columns:
  print(f"\n--- {col} ---")
  print(master_df[col].value_counts(dropna=False))


--- Sub_ID ---
Sub_ID
3b67b405-f160-42fe-a6cb-865444ce9e55    24
95542e9c-6757-4084-b6d2-3970426f3d9f    24
d172fa57-ad3f-4d13-97c7-ed9bbc7b4f68    24
43c4606e-257d-4788-90f0-74094d2fa9f2    24
b50fd8b5-8889-4945-84ff-c9ec6ef01759    24
                                        ..
9d53653d-f90e-461c-9203-b3264f0f3b62    24
4549a0d8-eb7d-424b-ba1c-963a07a161fc    24
ee1537f1-4d41-4736-a536-cb4e5d1c4b6f    24
2528b484-e3f6-4f67-871b-8bcc6dd3ed0a    24
2741dc40-f5e5-4b3b-8383-41d64c1e12c5    24
Name: count, Length: 30000, dtype: int64

--- Month_ID ---
Month_ID
2024-01    30000
2024-02    30000
2024-03    30000
2024-04    30000
2024-05    30000
2024-06    30000
2024-07    30000
2024-08    30000
2024-09    30000
2024-10    30000
2024-11    30000
2024-12    30000
2025-01    30000
2025-02    30000
2025-03    30000
2025-04    30000
2025-05    30000
2025-06    30000
2025-07    30000
2025-08    30000
2025-09    30000
2025-10    30000
2025-11    30000
2025-12    30000
Name: count, dtype: int64

-